In [1]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

# Download data

In [2]:
# data
download_file(
    url="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE247599&format=file",
    dest_path='../non_curated/song_2025_jurkat_hiv.tar',
    unarchive=True
)
# guide-cell mapping
download_file(
    url="https://raw.githubusercontent.com/davidliwei/PS/refs/heads/main/datasets/HIV_Perturb-seq/BARCODE_H13Ld2EGFP.txt",
    dest_path='../supplementary/song_2025_jurkat_hiv/BARCODE_H13Ld2EGFP.txt',
    unarchive=False
)

Downloaded https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE247599&format=file to ../non_curated/song_2025_jurkat_hiv.tar
File ../supplementary/song_2025_jurkat_hiv/BARCODE_H13Ld2EGFP.txt already exists. Skipping download.



gzip: stdin: not in gzip format
tar: Child returned status 1
tar: Error is not recoverable: exiting now


Data is in three sets of files (mtx, barcodes, features) for each of the three conditions (NoDrug, LRA-Negative (no HIV expression)) and LRA-Positive (HIV expression). 

In [17]:
import scanpy as sc
import scipy.io
import anndata as ad
import numpy as np

DATA_DIR = "../non_curated"

samples = {
    "LRA-Positive": "GSM7897841_JKLAT-LRA-Positive",
    "LRA-Negative": "GSM7897842_JKLAT-LRA-Negative",
    "NoDrug":       "GSM7897843_JKLAT-NoDrug",
}

def read_10x_mtx(data_dir, prefix):
    matrix   = scipy.io.mmread(f"{data_dir}/{prefix}_matrix.mtx.gz").T.tocsr()
    barcodes = pd.read_csv(f"{data_dir}/{prefix}_barcodes.tsv.gz", header=None)[0]
    features = pd.read_csv(
        f"{data_dir}/{prefix}_features.tsv.gz",
        sep="\t", header=None,
        names=["gene_id", "gene_name", "feature_type"],
    )
    obs = pd.DataFrame(index=barcodes)
    var = features.set_index("gene_id")
    return ad.AnnData(X=matrix, obs=obs, var=var)

adatas = {}
for condition, prefix in samples.items():
    adata = read_10x_mtx(DATA_DIR, prefix)
    adata.obs["condition"] = condition
    adatas[condition] = adata
    print(f"{condition}: {adata.shape}")

adata = ad.concat(adatas.values(), label="condition", keys=adatas.keys())
adata.var = adatas["LRA-Positive"].var  # restore var metadata lost during concat
print("\nConcatenated:", adata)


LRA-Positive: (2617899, 36636)
LRA-Negative: (2784659, 36636)
NoDrug: (2830966, 36636)

Concatenated: AnnData object with n_obs × n_vars = 8233524 × 36636
    obs: 'condition'
    var: 'gene_name', 'feature_type'


/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [18]:
# add cell barcode to obs
adata.obs['cell_barcode'] = adata.obs_names + '_' + adata.obs['condition'].astype(str)
adata.obs[['cell_barcode']]

,cell_barcode
0,
AAACCCAAGAAACACT-1,AAACCCAAGAAACACT-1_LRA-Positive
AAACCCAAGAAACCAT-1,AAACCCAAGAAACCAT-1_LRA-Positive
AAACCCAAGAAACCCA-1,AAACCCAAGAAACCCA-1_LRA-Positive
AAACCCAAGAAACCCG-1,AAACCCAAGAAACCCG-1_LRA-Positive
AAACCCAAGAAACTAC-1,AAACCCAAGAAACTAC-1_LRA-Positive
...,...
TTTGTTGTCTTTCTTC-1,TTTGTTGTCTTTCTTC-1_NoDrug
TTTGTTGTCTTTGATC-1,TTTGTTGTCTTTGATC-1_NoDrug
TTTGTTGTCTTTGCAT-1,TTTGTTGTCTTTGCAT-1_NoDrug


In [19]:
adata.obs

,condition,cell_barcode
0,,
AAACCCAAGAAACACT-1,LRA-Positive,AAACCCAAGAAACACT-1_LRA-Positive
AAACCCAAGAAACCAT-1,LRA-Positive,AAACCCAAGAAACCAT-1_LRA-Positive
AAACCCAAGAAACCCA-1,LRA-Positive,AAACCCAAGAAACCCA-1_LRA-Positive
AAACCCAAGAAACCCG-1,LRA-Positive,AAACCCAAGAAACCCG-1_LRA-Positive
AAACCCAAGAAACTAC-1,LRA-Positive,AAACCCAAGAAACTAC-1_LRA-Positive
...,...,...
TTTGTTGTCTTTCTTC-1,NoDrug,TTTGTTGTCTTTCTTC-1_NoDrug
TTTGTTGTCTTTGATC-1,NoDrug,TTTGTTGTCTTTGATC-1_NoDrug
TTTGTTGTCTTTGCAT-1,NoDrug,TTTGTTGTCTTTGCAT-1_NoDrug


In [20]:
adata.var

,gene_name,feature_type
gene_id,,
ENSG00000243485,MIR1302-2HG,Gene Expression
ENSG00000237613,FAM138A,Gene Expression
ENSG00000186092,OR4F5,Gene Expression
ENSG00000238009,AL627309.1,Gene Expression
ENSG00000239945,AL627309.3,Gene Expression
...,...,...
HSPD0000052096,HSPD0000052096_MAP3K14,CRISPR Guide Capture
HSPD0000052097,HSPD0000052097_MAP3K14,CRISPR Guide Capture
HSPD0000073810,HSPD0000073810_BRD4,CRISPR Guide Capture


# Add available guide information to adata.obs

In [21]:
guide_map_df = pd.read_csv("../supplementary/song_2025_jurkat_hiv/BARCODE_H13Ld2EGFP.txt", sep="\t")
guide_map_df['cell_barcode'] = guide_map_df['cell'].str.split('_').str[1]
guide_map_df['cell_barcode'] = guide_map_df['cell_barcode'] + '_' + guide_map_df['cell'].str.split('_').str[0].map({
    'pos': 'LRA-Positive',
    'nes': 'LRA-Negative',
    'nodrug': 'NoDrug'
})
# some cells have multiple guides - group by cell barcode and aggregate barcode, gene, sgrna together, separated by '|'
# then remove the cells with multiple guides
guide_map_df = guide_map_df.groupby('cell_barcode').agg({
    'barcode': lambda x: '|'.join(x),
    'gene': lambda x: '|'.join(x),
    'sgrna': lambda x: '|'.join(x)
}).reset_index()
# filter out cells with multiple guides
guide_map_df = guide_map_df[~guide_map_df['sgrna'].str.contains('\\|')]
# standardise controls
guide_map_df['gene'] = (
    guide_map_df['barcode']
    .replace(r'NegativeControl\d+', 'control_nontargeting', regex=True)
    .replace('HScontrol-AAVS1', 'control_gsh'))
# clean gene names
guide_map_df['gene'] = guide_map_df['gene'].str.split('-').str[1].fillna(guide_map_df['gene'])
# rename cols
guide_map_df = guide_map_df.rename(columns={'barcode': 'perturbation_name', 'sgrna': 'guide_sequence'})
guide_map_df

,cell_barcode,perturbation_name,gene,guide_sequence
0,AAACCCAAGCTTGTTG-1_NoDrug,HSPD0000033202-PRKCA,PRKCA,TTGTGAACGTTCATATCGC
1,AAACCCAAGGGTGGGA-1_LRA-Positive,HSPD0000046601-NELFE,NELFE,CCGGGAACGGGACAGGGAT
2,AAACCCAAGGTTCCGC-1_NoDrug,HSPD0000018545-HDAC2,HDAC2,ACAACAGATCGTGTAATGA
3,AAACCCAAGGTTGGAC-1_NoDrug,HSPD0000052097-MAP3K14,MAP3K14,CCACTTTCCGCAGAACACA
4,AAACCCACAGCTGTGC-1_LRA-Negative,HSPD0000002159-BIRC2,BIRC2,TGAAGACATCTCTTCATCG
...,...,...,...,...
21873,TTTGTTGGTGGTTTAC-1_LRA-Positive,HScontrol-AAVS1,control_gsh,GGGGCCACTAGGGACAGGAT
21874,TTTGTTGGTTCAGGTT-1_LRA-Positive,HSPD0000005747-CCNT1,CCNT1,TGGGTCGTGTTGTGACCAT
21876,TTTGTTGTCTCGCGTT-1_NoDrug,HSPD0000046601-NELFE,NELFE,CCGGGAACGGGACAGGGAT
21877,TTTGTTGTCTCTCCGA-1_LRA-Negative,HSPD0000028138-NFKBIA,NFKBIA,TGGACGACCGCCACGACAG


In [22]:
# merge guide map with adata.obs; 
adata.obs = adata.obs.merge(guide_map_df, on='cell_barcode', how='left')
adata = adata[~adata.obs['guide_sequence'].isna()].copy()  # filter out cells without guide assignment

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [23]:
adata.obs

,condition,cell_barcode,perturbation_name,gene,guide_sequence
457,LRA-Positive,AAACCCAAGGGTGGGA-1_LRA-Positive,HSPD0000046601-NELFE,NELFE,CCGGGAACGGGACAGGGAT
1193,LRA-Positive,AAACCCACAGGTTCGC-1_LRA-Positive,HSPD0000018544-HDAC2,HDAC2,GACTGATATGGCTGTTAAT
1503,LRA-Positive,AAACCCAGTAGGTAGC-1_LRA-Positive,HSPD0000002160-BIRC2,BIRC2,ACTCATTGCATAACTGTAG
1585,LRA-Positive,AAACCCAGTCACCACG-1_LRA-Positive,HSPD0000033203-PRKCA,PRKCA,AGGGGGCGGATTTACCTAA
1931,LRA-Positive,AAACCCAGTTAACAGA-1_LRA-Positive,HSPD0000033201-PRKCA,PRKCA,AAGCCCGTTTGGATCCATA
...,...,...,...,...,...
8231140,NoDrug,TTTGTTGAGTTATGGA-1_NoDrug,HSPD0000052097-MAP3K14,MAP3K14,CCACTTTCCGCAGAACACA
8231425,NoDrug,TTTGTTGCACATACTG-1_NoDrug,HSPD0000005747-CCNT1,CCNT1,TGGGTCGTGTTGTGACCAT
8232103,NoDrug,TTTGTTGGTATAGCTC-1_NoDrug,HSPD0000028140-NFKBIA,NFKBIA,GAAGTGATCCGCCAGGTGA
8232401,NoDrug,TTTGTTGGTGATTCAC-1_NoDrug,HSPD0000002159-BIRC2,BIRC2,TGAAGACATCTCTTCATCG


# Save to a non-curated h5ad file

In [24]:
adata.write_h5ad("../non_curated/h5ad/song_2025_jurkat_hiv.h5ad")
del(adata, guide_map_df, adatas)

... storing 'perturbation_name' as categorical
... storing 'gene' as categorical
... storing 'guide_sequence' as categorical


... storing 'gene_name' as categorical
... storing 'feature_type' as categorical


# Initialise the dataset object

In [25]:
noncurated_path = "../non_curated/h5ad/song_2025_jurkat_hiv.h5ad"
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/song_2025_jurkat_hiv.h5ad


In [26]:
cur_data.adata.obs

,condition,cell_barcode,perturbation_name,gene,guide_sequence
457,LRA-Positive,AAACCCAAGGGTGGGA-1_LRA-Positive,HSPD0000046601-NELFE,NELFE,CCGGGAACGGGACAGGGAT
1193,LRA-Positive,AAACCCACAGGTTCGC-1_LRA-Positive,HSPD0000018544-HDAC2,HDAC2,GACTGATATGGCTGTTAAT
1503,LRA-Positive,AAACCCAGTAGGTAGC-1_LRA-Positive,HSPD0000002160-BIRC2,BIRC2,ACTCATTGCATAACTGTAG
1585,LRA-Positive,AAACCCAGTCACCACG-1_LRA-Positive,HSPD0000033203-PRKCA,PRKCA,AGGGGGCGGATTTACCTAA
1931,LRA-Positive,AAACCCAGTTAACAGA-1_LRA-Positive,HSPD0000033201-PRKCA,PRKCA,AAGCCCGTTTGGATCCATA
...,...,...,...,...,...
8231140,NoDrug,TTTGTTGAGTTATGGA-1_NoDrug,HSPD0000052097-MAP3K14,MAP3K14,CCACTTTCCGCAGAACACA
8231425,NoDrug,TTTGTTGCACATACTG-1_NoDrug,HSPD0000005747-CCNT1,CCNT1,TGGGTCGTGTTGTGACCAT
8232103,NoDrug,TTTGTTGGTATAGCTC-1_NoDrug,HSPD0000028140-NFKBIA,NFKBIA,GAAGTGATCCGCCAGGTGA
8232401,NoDrug,TTTGTTGGTGATTCAC-1_NoDrug,HSPD0000002159-BIRC2,BIRC2,TGAAGACATCTCTTCATCG


### Standardise perturbation targets

In [27]:
cur_data.standardize_genes(
    slot='obs',
    input_column='gene',
    input_column_type='gene_symbol',
    multiple_entries=False
)

Mapping gene symbols: 100%|███████████████████████████████████████| 12/12 [00:00<00:00, 9807.41it/s]


--------------------------------------------------
Successfully mapped 12 out of 12 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: []
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Add `perturbed_target_number` column

In [28]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

Counted entries in column perturbed_target_symbol of adata.obs and stored in perturbed_target_number


### Encode chromosomes as integers

In [29]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


In [30]:
cur_data.adata.obs

,cell_barcode,guide_sequence,gene,perturbation_name,condition,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding
index,,,,,,,,,,,,,
0,AAACCCAAGGGTGGGA-1_LRA-Positive,CCGGGAACGGGACAGGGAT,NELFE,HSPD0000046601-NELFE,LRA-Positive,ENSG00000204356,NELFE,protein_coding,chr6:31952087-31959038;-1,6,457,1,6
1,AAACCCACAGGTTCGC-1_LRA-Positive,GACTGATATGGCTGTTAAT,HDAC2,HSPD0000018544-HDAC2,LRA-Positive,ENSG00000196591,HDAC2,protein_coding,chr6:113933028-114011308;-1,6,1193,1,6
2,AAACCCAGTAGGTAGC-1_LRA-Positive,ACTCATTGCATAACTGTAG,BIRC2,HSPD0000002160-BIRC2,LRA-Positive,ENSG00000110330,BIRC2,protein_coding,chr11:102347208-102378680;1,11,1503,1,11
3,AAACCCAGTCACCACG-1_LRA-Positive,AGGGGGCGGATTTACCTAA,PRKCA,HSPD0000033203-PRKCA,LRA-Positive,ENSG00000154229,PRKCA,protein_coding,chr17:66302613-66810743;1,17,1585,1,17
4,AAACCCAGTTAACAGA-1_LRA-Positive,AAGCCCGTTTGGATCCATA,PRKCA,HSPD0000033201-PRKCA,LRA-Positive,ENSG00000154229,PRKCA,protein_coding,chr17:66302613-66810743;1,17,1931,1,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19875,TTTGTTGAGTTATGGA-1_NoDrug,CCACTTTCCGCAGAACACA,MAP3K14,HSPD0000052097-MAP3K14,NoDrug,ENSG00000006062,MAP3K14,protein_coding,chr17:45263119-45317112;-1,17,8231140,1,17
19876,TTTGTTGCACATACTG-1_NoDrug,TGGGTCGTGTTGTGACCAT,CCNT1,HSPD0000005747-CCNT1,NoDrug,ENSG00000129315,CCNT1,protein_coding,chr12:48688458-48716998;-1,12,8231425,1,12
19877,TTTGTTGGTATAGCTC-1_NoDrug,GAAGTGATCCGCCAGGTGA,NFKBIA,HSPD0000028140-NFKBIA,NoDrug,ENSG00000100906,NFKBIA,protein_coding,chr14:35401079-35404749;-1,14,8232103,1,14


### Add treatment information

In [31]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['condition'].map({
    'LRA-Positive': 'HIV infection|phorbol 13-acetate 12-myristate|ionomycin',
    'LRA-Negative': 'HIV infection|phorbol 13-acetate 12-myristate|ionomycin',
    'NoDrug': 'HIV infection|dimethyl sulfoxide'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['condition'].map({
    'LRA-Positive': 'EFO:0000764|CHEBI:37537|CHEBI:63954',
    'LRA-Negative': 'EFO:0000764|CHEBI:37537|CHEBI:63954',
    'NoDrug': 'EFO:0000764|CHEBI:28262'
})

### Add metadata

In [32]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRn",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        "technical_replicate": None,
        "biological_replicate": None,
        # treatment
        # "treatment_label": None,
        # "treatment_id": None,
        # model system
        "model_system_label": "cell_line",
        "model_system_id": None,
        "tissue": "blood",
        "cell_line_label": "JURKAT cell",
        "cell_type_label": "T cell",
        "disease_label": "T-cell childhood acute lymphocytic leukemia|HIV infectious disease",
        "disease_id": "MONDO:0000871|MONDO:0005109",

        "timepoint": "P7DT16H0M0S",
        "species": "Homo sapiens",
        "sex_label": "male",
        "sex_id": None,
        "developmental_stage_label": "adolescent",
        "developmental_stage_id": None,

        "study_title": "Decoding heterogeneous single-cell perturbation responses",
        "study_uri": "https://doi.org/10.1038/s41556-025-01626-9",
        "study_year": 2025,
        "first_author": "Bicna Song",
        "last_author": "Wei Li",

        "experiment_title": "Focused CRISPRn Perturb-seq of HIV latency regulators in Jurkat HIV model cell line under three conditions: DMSO-treated, PMA/I GFP⁺ (HIV reactivated), PMA/I GFP⁻ (HIV latent)",
        "experiment_summary": """
            A focused Perturb-seq screen targeting 10 known regulators of HIV transcription was performed in a latently infected Jurkat T-cell model.
            Cells were cultured for 7 days, after which they were subjected to either DMSO or phorbol 13-acetate 12-myristate/ionomycin (PMA/I) treatment.
            After 16 hours of treatment, PMA/I-treated cells were sorted into GFP-positive (HIV reactivated) and GFP-negative (HIV latent) populations, and all three conditions were profiled by single-cell RNA-seq.
            """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_label": "SpCas9",
        "library_generation_method_id": "EFO:0022876",

        "enzyme_delivery_method_label": "lentivirus transduction",
        "enzyme_delivery_method_id": None,

        "library_delivery_method_label": "lentivirus transduction",
        "library_delivery_method_id": None,

        "enzyme_integration_state_label": "random locus integration",
        "enzyme_integration_state_id": None,

        "library_integration_state_label": "random locus integration",
        "library_integration_state_id": None,

        "enzyme_expression_control_label": "constitutive transgene expression",
        "enzyme_expression_control_id": None,

        "library_expression_control_label": "constitutive transgene expression",
        "library_expression_control_id": None,

        "library_name": "custom",
        "library_uri": None,

        "library_format_label": "pooled",
        "library_format_id": None,

        "library_scope_label": "focused",
        "library_scope_id": None,

        "library_perturbation_type_label": "knockout",
        "library_perturbation_type_id": None,

        "library_manufacturer": "Wei Li lab",
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "3",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_label": "high-dimensional assay",
        "readout_dimensionality_id": None,

        "readout_type_label": "transcriptomic",
        "readout_type_id": None,

        "readout_technology_label": "single-cell rna-seq",
        "readout_technology_id": None,

        "method_name_label": "Perturb-seq",
        "method_name_id": None,

        "method_uri": None,

        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime",
        "sequencing_library_kit_id": None,

        "sequencing_platform_label": "Illumina NovaSeq 6000",
        "sequencing_platform_id": None,

        "sequencing_strategy_label": "barcode sequencing",
        "sequencing_strategy_id": None,

        "software_counts_label": "CellRanger",
        "software_counts_id": None,

        "software_analysis_label": "Seurat",
        "software_analysis_id": None,

        "reference_genome_label": "GRCh38",
        "reference_genome_id": None,
        
        "license_label": "CC BY 4.0",
        "license_id": "SWO:1000065",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE247599",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE247599",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE247599_RAW.tar",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column technical_replicate added to adata.obs
Column biological_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_type_label added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_label added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_label added to adata.obs
Column developmental_stage_id added to adata.obs
Column study_title added to adata.obs
Colum

In [33]:
cur_data.adata.obs

,cell_barcode,guide_sequence,gene,perturbation_name,condition,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding,treatment_label,treatment_id,dataset_id,sample_id,perturbation_type_label,perturbation_type_id,data_modality,significant,significance_criteria,score_interpretation,technical_replicate,biological_replicate,model_system_label,model_system_id,tissue,cell_line_label,cell_type_label,disease_label,disease_id,timepoint,species,sex_label,sex_id,developmental_stage_label,developmental_stage_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_label,library_generation_method_id,enzyme_delivery_method_label,enzyme_delivery_method_id,library_delivery_method_label,library_delivery_method_id,enzyme_integration_state_label,enzyme_integration_state_id,library_integration_state_label,library_integration_state_id,enzyme_expression_control_label,enzyme_expression_control_id,library_expression_control_label,library_expression_control_id,library_name,library_uri,library_format_label,library_format_id,library_scope_label,library_scope_id,library_perturbation_type_label,library_perturbation_type_id,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_label,readout_dimensionality_id,readout_type_label,readout_type_id,readout_technology_label,readout_technology_id,method_name_label,method_name_id,method_uri,sequencing_library_kit_label,sequencing_library_kit_id,sequencing_platform_label,sequencing_platform_id,sequencing_strategy_label,sequencing_strategy_id,software_counts_label,software_counts_id,software_analysis_label,software_analysis_id,reference_genome_label,reference_genome_id,license_label,license_id,associated_datasets
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,AAACCCAAGGGTGGGA-1_LRA-Positive,CCGGGAACGGGACAGGGAT,NELFE,HSPD0000046601-NELFE,LRA-Positive,ENSG00000204356,NELFE,protein_coding,chr6:31952087-31959038;-1,6,457,1,6,HIV infection|phorbol 13-acetate 12-myristate|...,EFO:0000764|CHEBI:37537|CHEBI:63954,song_2025_jurkat_hiv,1,CRISPRn,None,Perturb-seq,None,None,None,None,None,cell_line,None,blood,JURKAT cell,T cell,T-cell childhood acute lymphocytic leukemia|HI...,MONDO:0000871|MONDO:0005109,P7DT16H0M0S,Homo sapiens,male,None,adolescent,None,Decoding heterogeneous single-cell perturbatio...,https://doi.org/10.1038/s41556-025-01626-9,2025,Bicna Song,Wei Li,Focused CRISPRn Perturb-seq of HIV latency reg...,\n A focused Perturb-seq screen tar...,12,19880,EFO:0022868,endogenous,SpCas9,EFO:0022876,lentivirus transduction,None,lentivirus transduction,None,random locus integration,None,random locus integration,None,constitutive transgene expression,None,constitutive transgene expression,None,custom,None,pooled,None,focused,None,knockout,None,Wei Li lab,2,3,34,None,high-dimensional assay,None,transcriptomic,None,single-cell rna-seq,None,Perturb-seq,None,None,10x Genomics Single Cell 3-prime,None,Illumina NovaSeq 6000,None,barcode sequencing,None,CellRanger,None,Seurat,None,GRCh38,None,CC BY 4.0,SWO:1000065,"[{""dataset_accession"": ""GSE247599"", ""dataset_u..."
1,AAACCCACAGGTTCGC-1_LRA-Positive,GACTGATATGGCTGTTAAT,HDAC2,HSPD0000018544-HDAC2,LRA-Positive,ENSG00000196591,HDAC2,protein_coding,chr6:113933028-114011308;-1,6,1193,1,6,HIV infection|phorbol 13-acetate 12-myristate|...,EFO:0000764|CHEBI:37537|CHEBI:63954,song_2025_jurkat_hiv,2,CRISPRn,None,Perturb-seq,None,None,None,None,None,cell_line,None,blood,JURKAT cell,T cell,T-cell childhood acute lymphocytic leukemia|HI...,MONDO:0000871|MONDO:0005109,P7DT16H0M0S,Homo sapiens,male,None,adolescent,None,Decoding he

### Curate tissue information


In [34]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower name_lower     ontology_id
0        blood              blood      blood  UBERON:0000178
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell type information

In [35]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 1 cell_type ontology terms from `cell_type_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower name_lower ontology_id
0       T cell             t cell     t cell  CL:0000084
--------------------------------------------------
Overwriting column cell_type_label in adata.obs


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell line information

In [36]:
cur_data.standardize_ontology(
    input_column='cell_line_label',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

Mapped 1 cell_line ontology terms from `cell_line_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower   name_lower  ontology_id
0  JURKAT cell        jurkat cell  jurkat cell  CLO:0007043
--------------------------------------------------
Overwriting column cell_line_label in adata.obs


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate disease information

In [37]:
# cur_data.standardize_ontology(
#     input_column='disease_label',
#     column_type='term_name',
#     ontology_type='disease',
#     overwrite=True
# )

### Match schema column order

In [38]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [39]:
cur_data.validate_data(slot='obs', verbose=True)

2026-04-27 14:31:59,585 INFO curation_tools.curation_tools: adata.obs is valid according to the obs_schema.


,dataset_id,sample_id,cell_barcode,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint,treatment_label,treatment_id,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,cell_line_label,cell_line_id,sex_label,sex_id,developmental_stage_label,developmental_stage_id,disease_label,disease_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,song_2025_jurkat_hiv,1,AAACCCAAGGGTGGGA-1_LRA-Positive,Perturb-seq,<NA>,<NA>,HSPD0000046601-NELFE,chr6:31952087-31959038;-1,6,6,1,ENSG00000204356,NELFE,protein_coding,CCGGGAACGGGACAGGGAT,CRISPRn,<NA>,P7DT16H0M0S,HIV infection|phorbol 13-acetate 12-myristate|...,EFO:0000764|CHEBI:37537|CHEBI:63954,<NA>,<NA>,cell_line,<NA>,Homo sapiens,blood,UBERON:0000178,T cell,CL:0000084,JURKAT cell,CLO:0007043,male,<NA>,adolescent,<NA>,T-cell childhood acute lymphocytic leukemia|HI...,MONDO:0000871|MONDO:0005109,Decoding heterogeneous single-cell perturbatio...,https://doi.org/10.1038/s41556-025-01626-9,2025,Bicna Song,Wei Li,Focused CRISPRn Perturb-seq of HIV latency reg...,\n A focused Perturb-seq screen tar...,12,19880,EFO:0022868,endogenous,EFO:0022876,SpCas9,<NA>,lentivirus transduction,<NA>,lentivirus transduction,<NA>,random locus integration,<NA>,random locus integration,<NA>,constitutive transgene expression,<NA>,constitutive transgene expression,custom,<NA>,<NA>,pooled,<NA>,focused,<NA>,knockout,Wei Li lab,2,3,34,<NA>,<NA>,high-dimensional assay,<NA>,transcriptomic,<NA>,single-cell rna-seq,<NA>,Perturb-seq,<NA>,<NA>,10x Genomics Single Cell 3-prime,<NA>,Illumina NovaSeq 6000,<NA>,barcode sequencing,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE247599"", ""dataset_u...",CC BY 4.0,SWO:1000065
1,song_2025_jurkat_hiv,2,AAACCCACAGGTTCGC-1_LRA-Positive,Perturb-seq,<NA>,<NA>,HSPD0000018544-HDAC2,chr6:113933028-114011308;-1,6,6,1,ENSG00000196591,HDAC2,protein_coding,GACTGATATGGCTGTTAAT,CRISPRn,<NA>,P7DT16H0M0S,HIV infection|phorbol 13-acetate 12-myristate|...,EFO:0000764|CHEBI:37537|CHEBI:63954,<NA>,<NA>,cell_line,<NA>,Homo sapiens,blood,UBERON:0000178,T cell,CL:0000084,JURKAT cell,CLO:0007043,male,<NA>,adolescent,<NA>,T-cell childhood acute lymphocytic leukemia|HI...,MONDO:0000871|MONDO:0005109,Decoding heterogeneous single-cell perturbatio...,https://doi.org/10.1038/s41556-

# VAR slot curation

### Standardise genes

In [40]:
cur_data.show_var()

Variable data:
DataFrame shape: (36636, 2)
--------------------------------------------------
                              gene_name          feature_type
gene_id                                                      
ENSG00000243485             MIR1302-2HG       Gene Expression
ENSG00000237613                 FAM138A       Gene Expression
ENSG00000186092                   OR4F5       Gene Expression
ENSG00000238009              AL627309.1       Gene Expression
ENSG00000239945              AL627309.3       Gene Expression
...                                 ...                   ...
HSPD0000052096   HSPD0000052096_MAP3K14  CRISPR Guide Capture
HSPD0000052097   HSPD0000052097_MAP3K14  CRISPR Guide Capture
HSPD0000073810      HSPD0000073810_BRD4  CRISPR Guide Capture
HSPD0000073811      HSPD0000073811_BRD4  CRISPR Guide Capture
HSPD0000073812      HSPD0000073812_BRD4  CRISPR Guide Capture

[36636 rows x 2 columns]
--------------------------------------------------


In [41]:
# Keep only feature_type == Gene Expression
cur_data.adata = cur_data.adata[:, cur_data.adata.var['feature_type'] == 'Gene Expression'].copy()

In [42]:
cur_data.create_columns(
    slot = 'var',
    col_dict={'gene_ensembl_id': cur_data.adata.var.index},
    overwrite=True
)

Column gene_ensembl_id added to adata.var


In [43]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ensembl_id",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

Missing Ensembl IDs: ['ENSG00000261757', 'ENSG00000228434', 'ENSG00000273403', 'ENSG00000253369', 'ENSG00000228008', 'ENSG00000286996', 'ENSG00000257603', 'ENSG00000273363', 'ENSG00000273164', 'ENSG00000272240', 'ENSG00000240401', 'ENSG00000280710', 'ENSG00000266450', 'ENSG00000263698', 'ENSG00000225655', 'ENSG00000238202', 'ENSG00000274367', 'ENSG00000261462', 'ENSG00000264334', 'ENSG00000253960', 'ENSG00000256694', 'ENSG00000240355', 'ENSG00000249860', 'ENSG00000224745', 'ENSG00000243944', 'ENSG00000269873', 'ENSG00000272482', 'ENSG00000250284', 'ENSG00000288019', 'ENSG00000251218', 'ENSG00000236673', 'ENSG00000112096', 'ENSG00000250046', 'ENSG00000274897', 'ENSG00000255028', 'ENSG00000261720', 'ENSG00000261737', 'ENSG00000230641', 'ENSG00000269899', 'ENSG00000276471', 'ENSG00000276631', 'ENSG00000255507', 'ENSG00000272948', 'ENSG00000261438', 'ENSG00000239332', 'ENSG00000232749', 'ENSG00000258631', 'ENSG00000233844', 'ENSG00000264920', 'ENSG00000255272', 'ENSG00000272196', 'ENSG0000

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Replace missing gene symbols with original gene names

In [44]:
cur_data.adata.var['gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(cur_data.adata.var['gene_name'])

### Remove non ENSG entries from ensembl_gene_id column

In [45]:
cur_data.adata.var.loc[~cur_data.adata.var['ensembl_gene_id'].str.startswith('ENSG'), 'ensembl_gene_id'] = np.nan

### Validate var metadata

In [46]:
cur_data.validate_data(slot='var')

2026-04-27 14:32:37,108 INFO curation_tools.curation_tools: adata.var is valid according to the var_schema.


,ensembl_gene_id,gene_symbol
index,,
0,ENSG00000243485,MIR1302-2HG
1,ENSG00000237613,FAM138A
2,ENSG00000186092,OR4F5
3,ENSG00000241860,AL627309.1
4,ENSG00000239945,AL627309.3
...,...,...
36597,ENSG00000278633,AC023491.2
36598,ENSG00000276017,AC007325.1
36599,ENSG00000278817,AC007325.4


# Save the dataset

In [47]:
cur_data.save_curated_data_h5ad()

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)
... storing 'dataset_id' as categorical


... storing 'data_modality' as categorical
... storing 'significance_criteria' as categorical
... storing 'perturbation_name' as categorical
... storing 'perturbed_target_coord' as categorical
... storing 'perturbed_target_chromosome' as categorical
... storing 'perturbed_target_ensg' as categorical
... storing 'perturbed_target_symbol' as categorical
... storing 'perturbed_target_biotype' as categorical
... storing 'guide_sequence' as categorical
... storing 'perturbation_type_label' as categorical
... storing 'perturbation_type_id' as categorical
... storing 'timepoint' as categorical
... storing 'treatment_label' as categorical
... storing 'treatment_id' as categorical
... storing 'technical_replicate' as categorical
... storing 'biological_replicate' as categorical
... storing 'model_system_label' as categorical
... storing 'model_system_id' as categorical
... storing 'species' as categorical
... storing 'tissue_label' as categorical
... storing 'tissue_id' as categorical
... stori

✅ Curated h5ad data saved to ../curated/h5ad/song_2025_jurkat_hiv_curated.h5ad


In [51]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/song_2025_jurkat_hiv_curated_metadata.parquet


# Upload to BigQuery

In [52]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/song_2025_jurkat_hiv_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file ../curated/parquet/song_2025_jurkat_hiv_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 19880 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [53]:
!gcloud storage cp ../curated/h5ad/song_2025_jurkat_hiv_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../curated/h5ad/song_2025_jurkat_hiv_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/song_2025_jurkat_hiv_curated.h5ad
  Completed files 24/1 | 1.1GiB/1.1GiB | 323.6MiB/s                            

Average throughput: 290.8MiB/s
